In [97]:
import pandas as pd

In [53]:
# Step 1: Read the CSV file
input_file = "summerOly_athletes.csv"  # Replace with your input file path
output_file = "summerOly_athletes_ID.csv"  # Replace with your output file path
df = pd.read_csv(input_file)

# Step 2: Map names to unique IDs
name_column = "Name"  # Replace with the column name for names
df["ID"] = df[name_column].astype('category').cat.codes
# move the ID column to the front, then drop the name column
df = df[["ID"] + [col for col in df.columns if col != "ID"]]

# Step 3: Save the new CSV file
df.to_csv(output_file, index=False)

print(f"Encoded names to IDs and saved to {output_file}.")

Encoded names to IDs and saved to summerOly_athletes_ID.csv.


In [113]:
def sort_by_name(file_path: str, output_path: str) -> None:

    df = pd.read_csv(file_path)

    # Step 1: Sort by name
    name_column = "Name"  # Replace with the column name for names
    if name_column not in df.columns:
        raise ValueError(f"'{name_column}' column is missing from the dataset.")
    
    df = df.sort_values(by=name_column)

    # Save the result to the output path
    df.to_csv(output_path, index=False)

def map_ids(file_path: str, output_path: str) -> None:
    df = pd.read_csv(file_path)

    # Step 2: Map names to unique IDs
    name_column = "Name"  # Replace with the column name for names
    year_column = "Year"  # Replace with the column name for years
    noc_column = "NOC"  # Replace with the column name for country codes
    if year_column not in df.columns or noc_column not in df.columns:
        raise ValueError(f"'{year_column}' or '{noc_column}' column is missing from the dataset.")
    
    # Sort the DataFrame by name, NOC, and year to make processing easier
    df = df.sort_values(by=[name_column, noc_column, year_column])

    # Create a new ID column
    current_id = 0
    previous_name = None
    previous_year = None
    previous_noc = None

    def assign_id(row):
        nonlocal current_id, previous_name, previous_year, previous_noc
        name, year, noc = row[name_column], row[year_column], row[noc_column]
        # Assign a new ID if the name, NOC, or year difference conditions are met
        if (name != previous_name or
            noc != previous_noc or
            (previous_year is not None and abs(year - previous_year) >= 36)):
            current_id += 1
        previous_name = name
        previous_year = year
        previous_noc = noc
        return current_id

    df["ID"] = df.apply(assign_id, axis=1)

    # Step 1: Rearrange columns
    # Move the ID column to the front, don't drop the name column
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index("ID")))
    df = df[cols]

    # Save the result to the output path
    df.to_csv(output_path, index=False)
    return df

In [114]:
input_file = "summerOly_athletes.csv"
df = pd.read_csv(input_file)
sort_by_name(input_file, "sorted.csv")
df = map_ids("sorted.csv", "test.csv")
# create 4 new columns for gold, silver, bronze, and no medal which is 0 or 1 based on column Medal
df["Gold"] = df["Medal"].apply(lambda x: 1 if x == "Gold" else 0)
df["Silver"] = df["Medal"].apply(lambda x: 1 if x == "Silver" else 0)
df["Bronze"] = df["Medal"].apply(lambda x: 1 if x == "Bronze" else 0)
df["No medal"] = df["Medal"].apply(lambda x: 1 if x == "No medal" else 0)

In [115]:
# Change South Korea NOC from KOR to SKR
df.loc[(df["NOC"] == "KOR") & (df["Year"] >= 1948), "NOC"] = "SKR"

In [116]:
def distribute_medals(df, former, current):
    medal_count = {country:{"Gold": 0, "Silver": 0, "Bronze": 0, "No medal": 0} for country in current}
    # Iterate through the rows of the current dataframe, and update the medal count according to column Medal if the NOC is in the medal_count dictionary
    for index, row in df.iterrows():
        if row["NOC"] in medal_count and row["Team"] != "N/A":
            medal_count[row["NOC"]][row["Medal"]] += 1
    # sum total medals for gold, silver, ... for all countries
    total_count = {"Gold": 0, "Silver": 0, "Bronze": 0, "No medal": 0}
    for country in medal_count:
       for medal in medal_count[country]:
           total_count[medal] += medal_count[country][medal]

    medal_ratio = {country:{"Gold": 0.0, "Silver": 0.0, "Bronze": 0.0, "No medal": 0.0} for country in current}
    for country in medal_count:
       for medal in medal_count[country]:
           medal_ratio[country][medal] = medal_count[country][medal]/total_count[medal]
    print(medal_ratio)

    df_copy = df.copy()
    new_rows = []
    for index, row in df_copy.iterrows():
        if row["NOC"] == former:
            # delete the row and directly below add 1 row for each country in medal_count. Same content except for the NOC
            df.drop(index, inplace=True)
            for country in medal_count:
                new_rows.append(
                    {"ID": row["ID"], 
                     "Name": row["Name"],
                     "Sex": row["Sex"],
                     "Team": "N/A",
                     "NOC": country, 
                     "Year": row["Year"],
                     "City": row["City"],
                     "Sport": row["Sport"],	
                     "Event": row["Event"],
                     "Medal": row["Medal"],
                     "Gold": row["Gold"] * medal_ratio[country]["Gold"], 
                     "Silver": row["Silver"] * medal_ratio[country]["Silver"], 
                     "Bronze": row["Bronze"] * medal_ratio[country]["Bronze"], 
                     "No medal": row["No medal"] * medal_ratio[country]["No medal"],
                     })
    # Convert the list of dictionaries to a DataFrame
    new_rows_df = pd.DataFrame(new_rows)

    # Use pd.concat to append the new rows
    df = pd.concat([df, new_rows_df], ignore_index=True)    
    return df

In [117]:
def combine_medals(df, former, current):
    df_copy = df.copy()
    new_rows = []
    for index, row in df_copy.iterrows():
        if row["NOC"] in former:
            # delete the row and directly below add 1 row for each country in medal_count. Same content except for the NOC
            df.drop(index, inplace=True)
            new_rows.append(
                {"ID": row["ID"], 
                "Name": row["Name"],
                "Sex": row["Sex"],
                "Team": "N/A",
                "NOC": current, 
                "Year": row["Year"],
                "City": row["City"],
                "Sport": row["Sport"],	
                "Event": row["Event"],
                "Medal": row["Medal"],
                "Gold": row["Gold"], 
                "Silver": row["Silver"], 
                "Bronze": row["Bronze"], 
                "No medal": row["No medal"],
                })

    # Convert the list of dictionaries to a DataFrame
    new_rows_df = pd.DataFrame(new_rows)

    # Use pd.concat to append the new rows
    df = pd.concat([df, new_rows_df], ignore_index=True)    
    return df

In [118]:
output_file = "summerOly_athletes_distribute_medals.csv"
distribute_dict = {
    "URS": ["RUS", "ARM", "AZE", "BLR", "EST", "GEO", "KAZ", "KGZ", "LAT", "LTU", "MDA", "TJK", "TKM", "UKR", "UZB"],
    "KOR": ["SKR", "PRK"],
    "YUG": ["SRB", "MNE", "CRO", "SLO", "BIH", "MKD", "KOS"],
    "TCH": ["CZE", "SVK"],
    "AIN": ["RUS", "BLR"]
}

for former, current in distribute_dict.items():
    df = distribute_medals(df, former, current)

df.to_csv(output_file, index=False)

{'RUS': {'Gold': 0.6505494505494506, 'Silver': 0.5504950495049505, 'Bronze': 0.4395750332005312, 'No medal': 0.2513138138138138}, 'ARM': {'Gold': 0.004395604395604396, 'Silver': 0.019801980198019802, 'Bronze': 0.01593625498007968, 'No medal': 0.017454954954954954}, 'AZE': {'Gold': 0.01978021978021978, 'Silver': 0.033663366336633666, 'Bronze': 0.04249667994687915, 'No medal': 0.029185435435435437}, 'BLR': {'Gold': 0.041758241758241756, 'Silver': 0.09108910891089109, 'Bronze': 0.09163346613545817, 'No medal': 0.11721096096096097}, 'EST': {'Gold': 0.02857142857142857, 'Silver': 0.019801980198019802, 'Bronze': 0.033200531208499334, 'No medal': 0.04420045045045045}, 'GEO': {'Gold': 0.02857142857142857, 'Silver': 0.027722772277227723, 'Bronze': 0.02656042496679947, 'No medal': 0.025713213213213213}, 'KAZ': {'Gold': 0.04395604395604396, 'Silver': 0.04950495049504951, 'Bronze': 0.05444887118193891, 'No medal': 0.09234234234234234}, 'KGZ': {'Gold': 0.0, 'Silver': 0.009900990099009901, 'Bronze':

In [119]:
input_file = "summerOly_athletes_distribute_medals.csv"
output_file = "summerOly_athletes_distribute_combine_medals.csv"
#df = pd.read_csv(input_file)
combine_dict = {
    "GER": ["FRG", "GDR"]
}
for current, former in combine_dict.items():
    df = combine_medals(df, former, current)

df.to_csv(output_file, index=False)

In [120]:
team_keyword_list = set(["Relay", "Double", "Team", "Mixed", "Two Person", "Three Person", "Four Person", "Five Person", "Six Person", "Pairs", "Fours", "Quadruple", "Sixes", "Eights",
                         "Basketball", "Baseball", "Volleyball", "Tandem", "Football", "Handball", "Hockey", "Curling", "Water Polo", "Rugby", "Rowing", "Soccer", "Softball", "Rugby", "Polo", "Cricket", "Lacrosse", "Tug-Of-War"])

In [121]:
input_file = "summerOly_athletes_distribute_combine_medals.csv"
output_file = "summerOly_athletes_distribute_combine_medals_team.csv"
#df = pd.read_csv(input_file)
# create a column named Team_Sport that is 1 if either column "Sport" or "Event" contains a team keyword in team_keyword_list, 0 otherwise
df["Team_Sport"] = df.apply(lambda row: 1 if any(keyword in row["Sport"] for keyword in team_keyword_list) or any(keyword in row["Event"] for keyword in team_keyword_list) else 0, axis=1)
df.to_csv(output_file, index=False)

In [122]:
host_dict = {
    "Athens": ["GRE"],
    "Athina": ["GRE"],
    "Paris": ["FRA"],
    "St. Louis": ["USA"],
    "London": ["GBR"],
    "Stockholm": ["SWE"],
    "Berlin": ["GER"],
    "Antwerp": ["BEL"],
    "Antwerpen": ["BEL"],
    "Chamonix": ["FRA"],
    "Paris": ["FRA"],
    "St. Moritz": ["SUI"],
    "Amsterdam": ["NED"],
    "Lake Placid": ["USA"],
    "Los Angeles": ["USA"],
    "Sapporo": ["JPN"],
    "Garmisch-Partenkirchen": ["GER"],
    "Tokyo": ["JPN"],
    "Helsinki": ["FIN"],
    "Cortina d'Ampezzo": ["ITA"],
    "Melbourne": ["AUS"],
    "Stockholm": ["SWE"],
    "Squaw Valley": ["USA"],
    "Rome": ["ITA"],
    "Roma": ["ITA"],
    "Innsbruck": ["AUT"],
    "Tokyo": ["JPN"],
    "Grenoble": ["FRA"],
    "Mexico City": ["MEX"],
    "Sapporo": ["JPN"],
    "Munich": ["GER"],
    "Innsbruck": ["AUT"],
    "Montreal": ["CAN"],
    "Lake Placid": ["USA"],
    "Moskva": ["RUS", "ARM", "AZE", "BLR", "EST", "GEO", "KAZ", "KGZ", "LAT", "LTU", "MDA", "TJK", "TKM", "UKR", "UZB"],
    "Moscow": ["RUS", "ARM", "AZE", "BLR", "EST", "GEO", "KAZ", "KGZ", "LAT", "LTU", "MDA", "TJK", "TKM", "UKR", "UZB"],
    "Sarajevo": ["SRB", "MNE", "CRO", "SLO", "BIH", "MKD", "KOS"],
    "Los Angeles": ["USA"],
    "Calgary": ["CAN"],
    "Seoul": ["SKR"],
    "Albertville": ["FRA"],
    "Barcelona": ["ESP"],
    "Lillehammer": ["NOR"],
    "Atlanta": ["USA"],
    "Nagano": ["JPN"],
    "Sydney": ["AUS"],
    "Salt Lake City": ["USA"],
    "Athens": ["GRE"],
    "Torino": ["ITA"],
    "Beijing": ["CHN"],
    "Vancouver": ["CAN"],
    "London": ["GBR"],
    "Sochi": ["RUS"],
    "Rio de Janeiro": ["BRA"],
    "Pyeongchang": ["SKR"],
    "Tokyo": ["JPN"],
    "Beijing": ["CHN"],
    "Paris": ["FRA"],
}

In [123]:
input_file = "summerOly_athletes_distribute_combine_medals_team.csv"
output_file = "summerOly_athletes_distribute_combine_medals_team_home.csv"
#df = pd.read_csv(input_file)
# create a column named Home that is 1 if the NOC is in the host_dict for the corresponding City and Year, 0 otherwise
df["Home"] = df.apply(lambda row: 1 if row["NOC"] in host_dict[row["City"]] else 0, axis=1)
df.to_csv(output_file, index=False)

In [124]:
df.tail(5)

,ID,Name,Sex,Team,NOC,Year,City,Sport,Event,Medal,Gold,Silver,Bronze,No medal,Team_Sport,Home
332959,131753,Zita-Eva Funkenhauser,F,N/A,GER,1984,Los Angeles,Fencing,"Fencing Women's Foil, Team",Gold,1.0,0.0,0.0,0.0,1,0
332960,131753,Zita-Eva Funkenhauser,F,N/A,GER,1988,Seoul,Fencing,"Fencing Women's Foil, Team",Gold,1.0,0.0,0.0,0.0,1,0
332961,131753,Zita-Eva Funkenhauser,F,N/A,GER,1988,Seoul,Fencing,"Fencing Women's Foil, Individual",Bronze,0.0,0.0,1.0,0.0,0,0
332962,132049,Zsolt-Georg Bhm,M,N/A,GER,1988,Seoul,Table Tennis,Table Tennis Men's Singles,No medal,0.0,0.0,0.0,1.0,0,0
332963,132049,Zsolt-Georg Bhm,M,N/A,GER,1988,Seoul,Table Tennis,Table Tennis Men's Doubles,No medal,0.0,0.0,0.0,1.0,1,0


In [194]:
def medal_count(df, history, file_path: str, output_path: str) -> None:

    # Step 1: Sort by NOC, sport, then by year
    df = df.sort_values(by=["NOC", "Sport", "Year"])
    df_copy = df.copy()

    medal_count = {"Gold":[0]*history, "Silver":[0]*history, "Bronze":[0]*history}
    athlete_count = [0]*history

    new_columns = []
    last_year = df_copy.iloc[0]["Year"]
    last_sport = df_copy.iloc[0]["Sport"]
    last_country = df_copy.iloc[0]["NOC"]
    for index, row in df_copy.iterrows():

        this_country = row["NOC"]
        this_sport = row["Sport"]
        this_year = row["Year"]

        if this_sport != last_sport or this_country != last_country:
            medal_count = {"Gold":[0]*history, "Silver":[0]*history, "Bronze":[0]*history}
            athlete_count = [0]*history
            last_sport = this_sport
            last_year = this_year
        
        if this_year - last_year >= 20:
            last_year += 4
            athlete_count.append(0)
            athlete_count.pop(0)
            for medal in medal_count:
                print("B", medal_count[medal])
                medal_count[medal].append(0)
                medal_count[medal].pop(0)
                print(medal_count[medal])
        for medal in medal_count:
            medal_count[medal][-1] += row[medal]
        athlete_count[-1] += 1
        new_columns.append([sum(medal_count["Gold"][0:-1]), sum(medal_count["Silver"][0:-1]), sum(medal_count["Bronze"][0:-1]), sum(athlete_count[0:-1])])
            

    # Convert the list of dictionaries to a DataFrame\
    new_columns_df = pd.DataFrame(new_columns, columns = ["Gold Tally", "Silver Tally", "Bronze Tally", "Athlete Tally"])
    new_columns_df.index = df.index
    print(new_columns_df.sum(0))

    # Use pd.concat to append the new rows
    df = pd.concat([df, new_columns_df], axis=1, ignore_index=True)    
    return df

In [195]:
file_path = "summerOly_athletes_distribute_combine_medals_team_home.csv"
output_path = "test_theta.csv"
df = pd.read_csv(file_path)
df = medal_count(df, 5, file_path, output_path)
df.to_csv(output_path, index=False)

/var/folders/x6/57ghq8v93_b4q7bp0mscrhqw0000gn/T/ipykernel_72452/2049075140.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


B [0, 0, 0, 0, 0.0]
[0, 0, 0, 0.0, 0]
B [0, 0, 0, 0, 0.0]
[0, 0, 0, 0.0, 0]
B [0, 0, 0, 0, 0.0]
[0, 0, 0, 0.0, 0]
B [0, 0, 0, 0.0, 0.0]
[0, 0, 0.0, 0.0, 0]
B [0, 0, 0, 0.0, 0.0]
[0, 0, 0.0, 0.0, 0]
B [0, 0, 0, 0.0, 0.0]
[0, 0, 0.0, 0.0, 0]
B [0, 0, 0.0, 0.0, 0.0]
[0, 0.0, 0.0, 0.0, 0]
B [0, 0, 0.0, 0.0, 0.0]
[0, 0.0, 0.0, 0.0, 0]
B [0, 0, 0.0, 0.0, 0.0]
[0, 0.0, 0.0, 0.0, 0]
B [0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0]
B [0

In [19]:
df.to_csv(output_path, index=False)